In [12]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

# Quantize ChatGLM3-7B using optimum and GPTQ

Ref:
 - [Quantize open LLMs using optimum and GPTQ](https://github.com/philschmid/deep-learning-pytorch-huggingface/blob/main/training/optimize-llama-2-gptq.ipynb)

## GPQT

GPTQ is a post-training quantziation method to compress LLMs, like GPT. GPTQ compresses GPT models by reducing the number of bits needed to store each weight in the model, from 32 bits down to just 3-4 bits. 

The main benefits are:

- Quantizes the weights of the model layer-by-layer to 4 bits instead of 16 bits, this reduces the needed memory by 4x.
- Quantization is done gradually to minimize the accuracy loss from quantization.
- Achieves same latency as fp16 model, but 4x less memory usage, sometimes faster due to custom kernels, e.g. Exllama
- Quantized weights can be saved to disk for a head of time quantization.

## 1. Setup Enviro

In [13]:
!pip install "transformers" "optimum" "auto-gptq" "accelerate" "safetensors" --upgrade

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.7 MB/s eta 0:00:0000:0100:01
  Using cached http://mirrors.aliyun.com/pypi/packages/90/e5/b22697903982284fe284568fb2663a2196694a8eee637f5cf4ccfe435a38/auto_gptq-0.7.1.tar.gz (126 kB)
  Preparing metadata (setup.py) ... done
Discarding http://mirrors.aliyun.com/pypi/packages/90/e5/b22697903982284fe284568fb2663a2196694a8eee637f5cf4ccfe435a38/auto_gptq-0.7.1.tar.gz#sha256=5c61ad380e9b4c603757c254765e9083a90a820cd0aff1b5d2c6f7fd96c85e80 (from http://mirrors.aliyun.com/pypi/simple/auto-gptq/) (requires-python:>=3.8.0): Requested auto-gptq from http://mirrors.aliyun.com/pypi/packages/90/e5/b22697903982284fe284568fb2663a2196694a8eee637f5cf4ccfe435a38/auto_gptq-0.7.1.tar.gz#sha256=5c61ad380e9b4c603757c254765e9083a90a820cd0aff1b5d2c6f7fd96c85e80 has inconsistent version: expected '0.7.1', but metadata has '0.7.1+cu1210'
  Using cached http://mirrors.aliyun.com/pypi/packages/34

## 2. Prepare Dataset for quantization

GPTQ is a post-training quantization method, so need to prepare a dataset to quantize our model. Here, use the `WikiText` dataset from the Hugging Face Hub. The dataset is used to quantize the weights to minimize the performance loss. It is recommended to use a quantization dataset with *at least 128 samples*.

In [2]:
dataset_id = 'wikitext2'

## 3. Load and Quantize Model

Optimum integrates GPTQ quantization in the `optimum.qptq` namespace with a `GPTQQuantizer`. The quantizer takes our dataset (id or list), bits, and model_seqlen as input. For more customization check [here](https://github.com/huggingface/optimum/blob/234a427450a7dcc978b227fa627ebcdab1764318/optimum/gptq/quantizer.py#L76).

In [3]:
from optimum.gptq import GPTQQuantizer

# GPTQ quantizer
quantizer = GPTQQuantizer(bits=4, dataset=dataset_id, model_seqlen=4096)
quantizer.quant_method = 'gptq'

Load model.

Use `THUDM/chatglm3-6b` and quantize it to 4bit.

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = 'THUDM/chatglm3-6b'

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForCausalLM.from_pretrained(model_id, low_cpu_mem_usage=True, torch_dtype=torch.float16, trust_remote_code=True)

The repository for THUDM/chatglm3-6b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/THUDM/chatglm3-6b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


config.json:   0%|          | 0.00/1.32k [00:00<?, ?B/s]

configuration_chatglm.py:   0%|          | 0.00/2.33k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/THUDM/chatglm3-6b:
- configuration_chatglm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_chatglm.py:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

...

model-00003-of-00007.safetensors:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

model-00005-of-00007.safetensors:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

model-00006-of-00007.safetensors:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

model-00007-of-00007.safetensors:   0%|          | 0.00/1.05G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [10]:
import os
import json

quantized_model = quantizer.quantize_model(model, tokenizer)

save_folder = './quantized_glm3/'
quantized_model.save_pretrained(save_folder, safe_serialization=True)

# load fresh, fast tokenizer and save it to disk
tokenizer = AutoTokenizer.from_pretrained(model_id).save_pretrained(save_folder)

# save quantize_config.json
with open(os.path.join(save_folder, 'quantize_config.json'), 'w', encoding='utf-8') as f:
    quantizer.disable_exllama = False
    json.dump(quantizer.to_dict(), f, indent=2)

RuntimeError: gptqmodel or auto-gptq is required in order to perform gptq quantzation: `pip install gptqmodel` or `pip install auto-gptq`. Please notice that auto-gptq will be deprecated in the future.

In [ ]:
with open(os.path.join(save_folder, "config.json"), "r", encoding="utf-8") as f:
  config = json.load(f)
  config["quantization_config"]["disable_exllama"] = False
  with open(os.path.join(save_folder, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

## 4. Test performance and inference speed

In [14]:
" Test NON-quantized model on a simple prompt"

import time

prompt = """
<|system|>
You are ChatGLM3, a large language model trained by Zhipu.AI. Follow the user's instructions carefully. Respond using markdown.
<|user|>
Use the following Input to create an instruction that could have been used to generate the input using an LLM.

Input:
Dear [boss name],

I'm writing to request next week, August 1st through August 4th, off as paid time off.

I have some personal matters to attend to that week that require me to be out of the office. I wanted to give you as much advance notice as possible so you can plan accordingly while I am away.

Thank you, [Your name]
<|assistant|>
"""

# helper function to generate text and measure latency
def generate_helper(pipeline, prompt=prompt):
    # warm up
    for i in range(5):
        _ = pipeline('warm up')

    # measure altency in a simple way
    start = time.time()
    out = pipeline(prompt, max_new_tokens=100, do_sample=True, top_p=0.9, temperature=0.9)
    end = time.time()

    generated_text = out[0]['generated_text'][len(prompt):]

    latency_per_token_in_ms = ((end - start) / len(pipepline.tokenizer(generated_text)['input_ids'])) * 1000
    return {
        "text": out[0]['generated_text'][len(prompt):],
        "latency": f"{round(latency_per_token_in_ms, 2)}ms/token"
    }

In [ ]:
" load the vanilla transformers model and run inference using the pipeline class "
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Hugging Face model id
model_id = "THUDM/chatglm3-6b-in"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16) # we load the model in fp16 on purpose

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
" create the vanilla base line "

import torch

vanilla_res = generate_helper(pipe)

print(f"Latency: {vanilla_res['latency']}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Generated Instruction: {vanilla_res['text']}")

# Latency: 37.49ms/token
# GPU memory: 12.62 GB
# Generated Instruction: 

In [ ]:
# clean up 
del pipe
del model 
del tokenizer
torch.cuda.empty_cache()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# path to gptq weights
model_id = './quantized_glm3/'

q_tokenizer = AutoTokenizer.from_pretrained(model_id)
q_model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)

qtq_pipe = pipeline("text-generation", model=q_model, tokenizer=q_tokenizer)

In [ ]:
gpq_res = generate_helper(qtq_pipe)

print(f"Latency: {gpq_res['latency']}")
print(f"GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Generated Instruction: {gpq_res['text']}")